# Attribute-Oriented Induction (AOI) & Bottom-Up Cubing (BUC)

**Author:** Rajath Gadagkar  
This notebook demonstrates concept-hierarchy generalization via **Attribute-Oriented Induction (AOI)** and iceberg data cube computation via **Bottom-Up Cubing (BUC)** — including in-memory, simulated out-of-memory paging, explicit Apriori pruning, and cardinality-sorted optimization — over a 10,000-record marketing campaign dataset.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path.cwd().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.mining.aoi import load_marketing_dataset, run_aoi
from src.mining.buc import buc_in_memory, buc_paged, buc_apriori_pruned, buc_iceberg_sorted
from src.mining.benchmark import run_full_benchmark

records = load_marketing_dataset(REPO_ROOT / 'data' / 'marketing' / 'marketing_campaigns.csv')
df = pd.DataFrame(records)
df[['Campaign_ID', 'Budget', 'Revenue_Generated', 'Subscription_Tier', 'Budget_Category', 'Revenue_Category']].head()

## 1. Attribute-Oriented Induction (AOI)

Continuous `Budget` and `Revenue_Generated` attributes are generalized into `Low`, `Medium`, and `High` concept tiers to extract high-level characteristic rules.

In [ ]:
aoi_result = run_aoi(records)
rules_df = pd.DataFrame(aoi_result['rules'])
display(rules_df)

pivot = rules_df.pivot(index='Budget_Category', columns='Revenue_Category', values='count')
plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='d', cmap='viridis')
plt.title('AOI Characteristic Rules Heatmap (Budget vs. Revenue)')
plt.tight_layout()
plt.show()

## 2. Bottom-Up Cube (BUC) & Performance Benchmarks

Execute in-memory BUC, out-of-memory paged BUC, Apriori-pruned BUC, and cardinality-sorted Iceberg BUC, and plot runtime sensitivity across `minsup` and `page_size`.

In [ ]:
bench = run_full_benchmark(records, repeats=3)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ms_df = pd.DataFrame(bench['minsup_sweep'])
axes[0].plot(ms_df['minsup'], ms_df['runtime_ms'], marker='o')
axes[0].set_title('Minimum Support (minsup) vs. Runtime')
axes[0].set_xlabel('minsup')
axes[0].set_ylabel('Median Runtime (ms)')
axes[0].grid(True, alpha=0.3)

var_df = pd.DataFrame(bench['variant_comparison'])
sns.barplot(data=var_df, x='runtime_ms', y='algorithm', ax=axes[1])
axes[1].set_title('BUC Variant Runtime Comparison (4 Dimensions)')
axes[1].set_xlabel('Median Runtime (ms)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()